# Reporte 03: Fuerza Capilar (Grado Nodal) y Nodos de Transferencia Intermodal ($FC$)

## 1. Justificación Teórica
El presente indicador evalúa la **Fuerza Capilar** de la red de transporte, definida topológicamente como el grado nodal ($k$) de cada estación. Este indicador cuantifica el nivel de convergencia de flujos y la capacidad de un nodo para actuar como punto de transferencia intermodal. 

Para obtener una radiografía urbana realista y mitigar la inflación geométrica, el modelo divide la Fuerza Capilar en dos dimensiones analíticas:

**A. Fuerza Capilar Geográfica (Macro-Hubs):**
Agrupa topológicamente estaciones físicamente colindantes (radio de tolerancia de 100m) en un subgrafo $H$. Consolida los flujos de sus estaciones constituyentes eliminando las duplicidades. Representa el fenómeno urbano del transbordo físico (ej. CETRAMs):

$$FC_{H} = \sum_{j \in H} \left( k_{in}(j) + k_{out}(j) \right) - \text{Aristas Internas}$$

**B. Fuerza Capilar Matemática / Estricta (Por Estación Individual):**
Calcula el grado de entrada y salida de un nodo topológico $i$, aplicando un filtro de frontera espacial (radio de 25m) para capturar únicamente los vectores de flujo. Representa la centralidad pura del nodo en el grafo:

$$FC_i = k_{in}(i)_{frontera} + k_{out}(i)_{frontera}$$

In [1]:
# ==========================================
# 2. Inicialización del Entorno y Librerías
# ==========================================
%load_ext autoreload
%autoreload 2
%matplotlib inline 

import sys
import os
import re
import ast
import folium
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Silenciar advertencias futuras para un reporte limpio
warnings.filterwarnings('ignore', category=FutureWarning)

# Asegurar que Jupyter encuentra la carpeta raíz del proyecto
proyecto_path = os.path.abspath('..')
if proyecto_path not in sys.path:
    sys.path.append(proyecto_path)

from src.infrastructure.go_client.client import fetch_full_network
from src.api.schemas.schemas import GeoJSONTransportSchema
from src.core.services.graph_builder import VFTGraphBuilder
from src.core.algorithms.topologicalIndicators.capillar_strength import CapillaryStrengthAnalyzer

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'

In [3]:
# ==========================================
# 3. Descarga de Red y Construcción del Grafo
# ==========================================
print("📥 Descargando red de transporte...")
raw_data = await fetch_full_network()

print("⚙️ Construyendo Grafo Dirigido VFT...")
validated_payload = GeoJSONTransportSchema(**raw_data)

plt.ioff() 
GRAF= VFTGraphBuilder(validated_payload)
G = GRAF.build_graph()
plt.close('all')
plt.ion()

print(f"✅ Grafo construido: {G.number_of_nodes()} nodos topológicos.")

📥 Descargando red de transporte...
2026-05-31 23:43:13 | INFO     | VFT_Model | Construyendo el puente hacia el módulo espacial de Go...
2026-05-31 23:43:13 | INFO     | VFT_Model | Solicitando capa espacial a: http://localhost:8080/movilidad/mapas/geojsonEstacion


2026-05-31 23:43:13 | INFO     | VFT_Model | Solicitando capa espacial a: http://localhost:8080/movilidad/mapas/geojsonLinea
2026-05-31 23:43:13 | INFO     | VFT_Model | Red extraída: 22766 entidades espaciales listas para VFT.
⚙️ Construyendo Grafo Dirigido VFT...
2026-05-31 23:43:13 | INFO     | VFT_Model | Iniciando VFTGraphBuilder en modo: REALISTIC_INTEGRATION
2026-05-31 23:43:13 | INFO     | VFT_Model | Fase 1 y 2: Extrayendo nodos y trazos base...
2026-05-31 23:43:16 | INFO     | VFT_Model | Fase 2 completada: 64459 aristas interestación creadas.
2026-05-31 23:43:16 | INFO     | VFT_Model | Grafo Base construido: 11115 Nodos y 32103 Segmentos.
2026-05-31 23:43:16 | WARNING  | VFT_Model | Eliminadas 1543 aristas phantom (distancia > 5 km). Causa probable: geometría MultiLineString fragmentada en el backend.
2026-05-31 23:43:16 | INFO     | VFT_Model | Fase 3: Integrando sistemas (Tolerancia peatonal: 85.0m)...
2026-05-31 23:43:44 | INFO     | VFT_Model | Se crearon 20482 aristas 

## 4. Ejecución del Motor Topológico
A continuación, instanciamos el analizador sobre el grafo dirigido (`G`) y calculamos ambos escenarios de fuerza capilar.

In [4]:
# ==========================================
# 4. Cálculo de Indicadores Capilares
# ==========================================
from src.core.algorithms.topologicalIndicators.capillar_strength import CapillaryStrengthAnalyzer
analizador_topologico = CapillaryStrengthAnalyzer(G)

# Estos dos DataFrames alimentarán TODO el resto del notebook
df_estricto = analizador_topologico.calculate_capillary_strength(snap_tolerance_m=25.0)
df_macro_hubs = analizador_topologico.calculate_geo_capillary_strength(tolerance_m=100.0, snap_tolerance_m=25.0)

print(f"📊 Nodos individuales procesados: {len(df_estricto)}")
print(f"📊 Macro-Hubs consolidados: {len(df_macro_hubs)}")

📊 Nodos individuales procesados: 11115
📊 Macro-Hubs consolidados: 4697


<hr/>

## 5. Análisis Geográfico: Macro-Hubs Intermodales
Este enfoque revela la infraestructura física de transbordo. Al agrupar estaciones cercanas, emergen los grandes polígonos de movilidad de la ciudad.

In [5]:
# Quitar el límite de caracteres para que los nombres se vean completos
pd.set_option('display.max_colwidth', None) 

# Si además tienes muchas columnas y Pandas oculta las de en medio, usa:
pd.set_option('display.max_columns', None)

# Es mejor usar display() en Jupyter en lugar de solo llamar al dataframe
display(df_macro_hubs.head(5))

,Macro_Hub,Sistemas_Integrados,Detalle_Estaciones,Estaciones_Agrupadas,Conexiones_Entrada,Conexiones_Salida,Fuerza_Capilar_Total
2174,Chimalcoyotl - Coapa - Hospital Manuel Gea González. - Instituto Nacional de Especialidades - San Fernando - San Fernando - Calz. de Tlalpan - Sanborns - Hospitales - Tlalpan - Allende - Tlalpan - Chimalcoyotl - Tlalpan - Coapa - Tlalpan - INER - Tlalpan - San Fernando - Tlalpan - San Fernando Cent. - Tlalpan - San Fernando Hospital General - Tlalpan - San Fernando Izq. INER - Tlalpan e INER - Tlalpan y Allende - Tlalpan y San Fernando - Unidad de Urgencias Respiratorias,"[CC, RTP]","{'RTP': ['Chimalcoyotl', 'Coapa', 'Hospital Manuel Gea González.', 'Instituto Nacional de Especialidades', 'San Fernando', 'San Fernando - Calz. de Tlalpan', 'Sanborns - Hospitales', 'Unidad de Urgencias Respiratorias'], 'CC': ['Tlalpan - Allende', 'Tlalpan - Chimalcoyotl', 'Tlalpan - Coapa', 'Tlalpan - INER', 'Tlalpan - San Fernando', 'Tlalpan - San Fernando Cent.', 'Tlalpan - San Fernando Hospital General', 'Tlalpan - San Fernando Izq. INER', 'Tlalpan e INER', 'Tlalpan y Allende', 'Tlalpan y San Fernando']}",21,66,64,130
136,Acoxpa - Acoxpa - Tlalpan - Acueducto - San Juan de Dios - Apatlaco - Calz. México-Xochimilco - Calz. de Tlalpan - Calz. de Tlalpan - Acoxpa - Calz. de Tlalpan - Calz. Acoxpa - Calz. de Tlalpan - Don Juan Bosco - Calz. de Tlalpan - Estadio Azteca - Calz. de Tlalpan - Glorieta Huipulco - Calz. de Tlalpan - Glorieta de Huipulco - Calz. de Tlalpan - México-Xochimilco - Calz. de Tlalpan - Renato Leduc - Calz. de Tlalpan - San Alejandro - Cetram Estadio Azteca Anden A - Cierre de Circuito Cetram Estadio Azteca - Elkar - Estadio Azteca - Estadio Azteca 2 - Huipulco - Luis Murillo - Luis Murillo - Tlalpan - Renato Leduc - Renato Leduc - Tlalpan - Renato Leduc y Calz. de Tlalpan - San Alejandro - Sobre Calz. Acoxpa - Sobre Calz. Acoxpa y Calz. De Tlalpan - Sobre Calz. De Tlalpan - Sobre Calz. De Tlalpan y Av. Acoxpa - Sobre Calz. De Tlalpan y Av. Acueducto - Sobre Calz. De Tlalpan y Av. Del Imán - Sobre Calz. De Tlalpan y Cto. Estadio Azteca - Sobre Calz. De Tlalpan y Estadio Azteca - Sobre Calz. De Tlalpan y La Paz - Sobre Calz. De Tlalpan y Luis Murillo - Sobre Calz. De Tlalpan y antes de Estadio Azteca - Sobre Calz. Tlalpan y Clínica 7 del IMSS - Sobre Calz. Tlalpan y E. Azteca - Sobre Calz. de Tlalpan e Instituto Electoral - Tlalpan - Tlalpan - Acoxpa - Tlalpan - Apatlaco - Tlalpan - Av. del Iman - Tlalpan - CFE y Deportivo - Tlalpan - Calz. México-Xochimilco - Tlalpan - Estadio Azteca - Tlalpan - Glorieta de Huipulco - Tlalpan - Huipulco Tren Ligero - Tlalpan - Renato Leduc - Tlalpan - San Juan Bosco - Tlalpan y Apatlaco - Tlalpan y Estadio Azteca - Tlalpan y Glorieta Huipulco - Tlalpan y Luis Murillo - Tlalpan y San Alejandro,"[CC, RTP, TL]","{'CC': ['Calz. de Tlalpan - México-Xochimilco', 'Luis Murillo - Tlalpan', 'Renato Leduc - Tlalpan', 'Renato Leduc y Calz. de Tlalpan', 'Sobre Calz. Acoxpa', 'Sobre Calz. Acoxpa y Calz. De Tlalpan', 'Sobre Calz. De Tlalpan', 'Sobre Calz. De Tlalpan y Av. Acoxpa', 'Sobre Calz. De Tlalpan y Av. Acueducto', 'Sobre Calz. De Tlalpan y Av. Del Imán', 'Sobre Calz. De Tlalpan y Cto. Estadio Azteca', 'Sobre Calz. De Tlalpan y Estadio Azteca', 'Sobre Calz. De Tlalpan y La Paz', 'Sobre Calz. De Tlalpan y Luis Murillo', 'Sobre Calz. De Tlalpan y antes de Estadio Azteca', 'Sobre Calz. Tlalpan y Clínica 7 del IMSS', 'Sobre Calz. Tlalpan y E. Azteca', 'Sobre Calz. de Tlalpan e Instituto Electoral', 'Tlalpan - Acoxpa', 'Tlalpan - Apatlaco', 'Tlalpan - Av. del Iman', 'Tlalpan - CFE y Deportivo', 'Tlalpan - Calz. México-Xochimilco', 'Tlalpan - Estadio Azteca', 'Tlalpan - Glorieta de Huipulco', 'Tlalpan - Huipulco Tren Ligero', 'Tlalpan - Renato Leduc', 'Tlalpan - San Juan Bosco', 'Tlalpan y Apatlaco', 'Tlalpan y Estadio Azteca', 'Tlalpan y Glorieta Huipulco', 'Tlalpan y Luis Murillo', 'Tlalpan y San Alejandro'], 'RTP': ['Acoxpa', 'Acoxpa - Tlalpan', 'Acueducto - San Juan de D

In [ ]:
# --- Gráfica de Barras ---
top_10_hubs = df_macro_hubs.head(10).copy()
top_10_hubs['Macro_Hub_Corto'] = top_10_hubs['Macro_Hub'].apply(lambda x: x[:45] + '...' if len(x) > 45 else x)

fig, ax = plt.subplots(figsize=(14, 6))

sns.barplot(
    data=top_10_hubs, 
    x='Fuerza_Capilar_Total', 
    y='Macro_Hub_Corto', 
    palette='viridis', 
    ax=ax
)

ax.set_title('Top 10 Macro-Hubs Geográficos', fontsize=14, fontweight='bold')
ax.set_xlabel('Fuerza Capilar Total ($FC_H$)')
ax.set_ylabel('')

for i in ax.containers: 
    ax.bar_label(i, padding=5, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- Mapa Interactivo ---
mapa_geo = folium.Map(location=[19.4326, -99.1332], zoom_start=11, tiles="CartoDB positron")

# Para mapear los Hubs, buscamos sus estaciones constituyentes en el df_estricto
for _, hub in top_10_hubs.iterrows():
    nodos_hub = df_estricto[df_estricto['Estacion'].apply(lambda x: x in hub['Macro_Hub'])]
    
    for _, nodo in nodos_hub.iterrows():
        try:
            lon, lat = ast.literal_eval(nodo['Nodo_ID'])
            folium.CircleMarker(
                location=(lat, lon), radius=max(hub['Fuerza_Capilar_Total']*0.1, 5),
                color="#2A7686", fill=True, fill_color="#2A7686", fill_opacity=0.6,
                popup=f"<b>Macro-Hub:</b> {hub['Macro_Hub_Corto']}<br><b>FC Geográfica:</b> {hub['Fuerza_Capilar_Total']}"
            ).add_to(mapa_geo)
        except: pass

display(mapa_geo)

<hr/>

## 6. Análisis Matemático: Centralidad Estricta
Evalúa la convergencia puntual sin agrupaciones, detectando puntos específicos (coordenadas únicas) que soportan una gran carga de vectores vehiculares.

In [8]:
# Es mejor usar display() en Jupyter en lugar de solo llamar al dataframe
display(df_estricto.head(5))

,Nodo_ID,Estacion,Sistemas_Integrados,Detalle_Estaciones,Estaciones_Agrupadas,Conexiones_Entrada,Conexiones_Salida,Fuerza_Capilar_Total
7270,"(-99.15056, 19.29909)",Luis Murillo,[RTP],{'RTP': ['Luis Murillo']},1,21,20,41
2817,"(-99.15056, 19.29858)",Calz. de Tlalpan - Glorieta Huipulco,[RTP],{'RTP': ['Calz. de Tlalpan - Glorieta Huipulco']},1,19,15,34
10211,"(-99.15071309, 19.29903108)",Sobre Calz. De Tlalpan y Luis Murillo,[CC],{'CC': ['Sobre Calz. De Tlalpan y Luis Murillo']},1,16,17,33
10717,"(-99.15054, 19.29906)",Tlalpan y Luis Murillo,[CC],{'CC': ['Tlalpan y Luis Murillo']},1,18,15,33
5540,"(-99.14703, 19.30132)",Estadio Azteca 2,[RTP],{'RTP': ['Estadio Azteca 2']},1,14,18,32


In [ ]:
# --- Gráfica de Barras ---
top_10_nodos = df_estricto.head(10).copy()

top_10_nodos['Estacion_Corta'] = top_10_nodos['Estacion'].apply(
    lambda x: x[:45] + '...' if len(x) > 45 else x
)
fig, ax = plt.subplots(figsize=(14, 6))

sns.barplot(
    data=top_10_nodos, 
    x='Fuerza_Capilar_Total', 
    y='Estacion_Corta', 
    palette='magma', 
    ax=ax
)

ax.set_title('Top 10 Nodos Individuales (Matemático)', fontsize=14, fontweight='bold')
ax.set_xlabel('Fuerza Capilar Total ($FC_i$)')
ax.set_ylabel('')

for i in ax.containers: 
    ax.bar_label(i, padding=5, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- Mapa Interactivo ---
mapa_math = folium.Map(location=[19.4326, -99.1332], zoom_start=11, tiles="CartoDB positron")

for _, row in top_10_nodos.iterrows():
    try:
        lon, lat = ast.literal_eval(row['Nodo_ID'])
        folium.CircleMarker(
            location=(lat, lon), radius=max(row['Fuerza_Capilar_Total']*0.2, 5),
            color="#C73E5C", fill=True, fill_color="#C73E5C", fill_opacity=0.7,
            popup=f"<b>Nodo:</b> {row['Estacion']}<br><b>FC Matemática:</b> {row['Fuerza_Capilar_Total']}"
        ).add_to(mapa_math)
    except: pass

display(mapa_math)

<hr/>

### 6.1. Distribución de la Capilaridad Matemática (Ley de Potencias)
La topología de la red sigue una distribución de **Scale-Free Network** (Ley de Potencias). 
¿Por qué ocurre esto? En los sistemas de transporte reales, la inmensa mayoría de las paradas actúan como nodos de paso simples (1 conexión de entrada y 1 de salida = FC de 2). Por el contrario, un grupo extremadamente reducido de nodos actúa como absorbedores masivos de rutas (decenas de conexiones). Esta desigualdad genera la curva logarítmica descendente que vemos a continuación.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(
    df_estricto['Fuerza_Capilar_Total'], 
    bins=40, 
    kde=True, 
    color='#C73E5C', 
    ax=ax
)
ax.set_title('Distribución de Fuerza Capilar (Scale-Free Network)', fontweight='bold')
ax.set_xlabel('Fuerza Capilar Total ($FC_i$)')
ax.set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# --- Tabla Demostrativa del Fenómeno ---
print("Ejemplo de la polarización en la red:")
df_ejemplo = pd.concat([df_estricto.head(2), df_estricto[df_estricto['Fuerza_Capilar_Total'] == 2].tail(2)])
_cols_pol = ['Estacion', 'Sistemas_Integrados', 'Fuerza_Capilar_Total']
display(df_ejemplo[_cols_pol])

<hr/>

## 7. Anexo: Diagnóstico Especulativo para Ruta Anillar (Periférico)
Este apartado aísla los nodos y Macro-Hubs que cruzan o se nombran "Periférico". Se divide en la perspectiva Matemática y Geográfica para identificar anclajes potenciales.

In [13]:
# Expresión regular para atrapar variantes de la palabra
patron = re.compile(r'perif[eé]rico', re.IGNORECASE)

df_peri_math = df_estricto[df_estricto['Estacion'].str.contains(patron, na=False)].sort_values(by='Fuerza_Capilar_Total', ascending=False)
df_peri_geo = df_macro_hubs[df_macro_hubs['Macro_Hub'].str.contains(patron, na=False)].sort_values(by='Fuerza_Capilar_Total', ascending=False)

### 7.1. Eje Periférico: Perspectiva Geográfica (Macro-Hubs)
Muestra los polígonos de transferencia más fuertes a lo largo de la vialidad.

In [ ]:
print("Ejemplo de Estaciones Periféricas en la Red Geográfica:")
_cols_geo = ['Macro_Hub', 'Sistemas_Integrados', 'Estaciones_Agrupadas', 'Conexiones_Entrada', 'Conexiones_Salida', 'Fuerza_Capilar_Total']
display(df_peri_geo.head(5))

In [ ]:
# --- Datos ---
top_peri_geo = df_peri_geo.head(10).copy()
top_peri_geo['Macro_Hub_Corto'] = top_peri_geo['Macro_Hub'].apply(
    lambda x: x[:45] + '...' if len(x) > 45 else x
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=top_peri_geo,
    x='Fuerza_Capilar_Total', 
    y='Macro_Hub_Corto',
    palette='crest',
    ax=ax
)

ax.set_title('Top 10 Macro-Hubs en Periférico', fontsize=14, fontweight='bold')
ax.set_xlabel('Fuerza Capilar Geográfica ($FC_H$)')
ax.set_ylabel('')

for i in ax.containers:
    ax.bar_label(i, padding=5, fontweight='bold')
    
plt.tight_layout()
plt.show()

In [ ]:
# Mapa
mapa_peri_geo = folium.Map(location=[19.3500, -99.1900], zoom_start=11, tiles="CartoDB positron")
for _, hub in top_peri_geo.iterrows():
    nodos_hub = df_estricto[df_estricto['Estacion'].apply(lambda x: x in hub['Macro_Hub'])]
    for _, nodo in nodos_hub.iterrows():
        try:
            lon, lat = ast.literal_eval(nodo['Nodo_ID'])
            folium.CircleMarker(
                location=(lat, lon), radius=max(hub['Fuerza_Capilar_Total']*0.2, 6),
                color="#1B4F72", fill=True, fill_opacity=0.7, popup=f"Hub: {hub['Macro_Hub_Corto']}"
            ).add_to(mapa_peri_geo)
        except: pass

display(mapa_peri_geo)

### 7.2. Eje Periférico: Perspectiva Matemática (Nodos Individuales)
Muestra las coordenadas exactas de mayor tensión vehicular intermodal sobre el anillo.

In [ ]:
_cols_peri_math = ['Nodo_ID', 'Estacion', 'Sistemas_Integrados', 'Conexiones_Entrada', 'Conexiones_Salida', 'Fuerza_Capilar_Total']
display(df_peri_math.head(5))

In [ ]:
# --- Datos ---
top_peri_math = df_peri_math.head(10).copy()
top_peri_math['Estacion_Corta'] = top_peri_math['Estacion'].apply(
    lambda x: x[:45] + '...' if len(x) > 45 else x
)

fig, ax = plt.subplots(figsize=(12, 5))

sns.barplot(
    data=top_peri_math, 
    x='Fuerza_Capilar_Total', 
    y='Estacion_Corta', 
    palette='flare', 
    ax=ax
)

ax.set_title('Top 10 Nodos Matemáticos en Periférico', fontsize=14, fontweight='bold')
ax.set_xlabel('Fuerza Capilar Matemática ($FC_i$)')
ax.set_ylabel('')

for i in ax.containers:
    ax.bar_label(i, padding=5, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Mapa
mapa_peri_math = folium.Map(location=[19.3500, -99.1900], zoom_start=11, tiles="CartoDB positron")
for _, row in top_peri_math.iterrows():
    try:
        lon, lat = ast.literal_eval(row['Nodo_ID'])
        folium.CircleMarker(
            location=(lat, lon), radius=max(row['Fuerza_Capilar_Total']*0.3, 6),
            color="#E67E22", fill=True, fill_opacity=0.8, popup=f"Nodo: {row['Estacion_Corta']}"
        ).add_to(mapa_peri_math)
    except: pass

display(mapa_peri_math)

<hr/>
**Nota Aclaratoria:** *Los mapas mostrados en la Sección 7 representan únicamente una evaluación de la infraestructura preexistente. Aún no se traza ni se mapea el recorrido geométrico propuesto para el "Corredor Anillar", por lo que esta sección se mantiene en un plano puramente especulativo para identificar los puntos naturales de anclaje de la futura línea.*

In [20]:
# ==========================================
# EXPORTACIÓN GIS — Fuerza Capilar → QGIS
# ==========================================
import geopandas as gpd
import ast
from shapely.geometry import Point
from pathlib import Path

EXPORT_PATH = Path("/home/galigaribaldi/Documentos/Code/Transport-gis-zmvm-mjg/data/processed/FuerzaCapilar.gpkg")

# Capa 1: Nodos individuales con FC >= 3 (filtra el ruido de nodos triviales)
puntos = []
for _, row in df_estricto[df_estricto['Fuerza_Capilar_Total'] >= 3].iterrows():
    try:
        lon, lat = ast.literal_eval(row['Nodo_ID'])
        sistemas = ', '.join(row['Sistemas_Integrados']) if isinstance(row['Sistemas_Integrados'], list) else str(row['Sistemas_Integrados'])
        puntos.append({
            'estacion': str(row['Estacion']),
            'sistemas': sistemas,
            'fc_total': int(row['Fuerza_Capilar_Total']),
            'geometry': Point(lon, lat)
        })
    except:
        pass

gdf_fc = gpd.GeoDataFrame(puntos, crs="EPSG:4326").to_crs("EPSG:32614")
gdf_fc.to_file(EXPORT_PATH, layer="fc_puntos", driver="GPKG", engine="pyogrio")
print(f"✅ fc_puntos: {len(gdf_fc)} nodos exportados")

# Capa 2: Top 20 Macro-Hubs (para etiquetas de los más críticos)
hubs = []
for _, hub in df_macro_hubs.head(20).iterrows():
    nodos_hub = df_estricto[df_estricto['Estacion'].apply(lambda x: x in hub['Macro_Hub'])]
    if not nodos_hub.empty:
        try:
            lon, lat = ast.literal_eval(nodos_hub.iloc[0]['Nodo_ID'])
            sistemas = ', '.join(hub['Sistemas_Integrados']) if isinstance(hub['Sistemas_Integrados'], list) else str(hub['Sistemas_Integrados'])
            hubs.append({
                'hub_nombre': hub['Macro_Hub'][:60],
                'sistemas': sistemas,
                'n_estaciones': int(hub['Estaciones_Agrupadas']),
                'fc_total': int(hub['Fuerza_Capilar_Total']),
                'geometry': Point(lon, lat)
            })
        except:
            pass

gdf_hubs = gpd.GeoDataFrame(hubs, crs="EPSG:4326").to_crs("EPSG:32614")
gdf_hubs.to_file(EXPORT_PATH, layer="fc_top20_hubs", driver="GPKG", engine="pyogrio")
print(f"✅ fc_top20_hubs: {len(gdf_hubs)} macro-hubs exportados")


✅ fc_puntos: 10537 nodos exportados
✅ fc_top20_hubs: 20 macro-hubs exportados
